In [17]:


import os
import warnings
warnings.filterwarnings('ignore')

# Data manipulation and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, accuracy_score,
                             precision_recall_curve, f1_score)

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [18]:
print("\n[STEP 1] Downloading dataset from Kaggle...")

# Check if kaggle is installed
try:
    import kaggle
    print("✓ Kaggle API is installed")
except ImportError:
    print("Installing Kaggle API...")
    os.system("pip install kaggle --break-system-packages -q")
    import kaggle

# Download the dataset
dataset_name = "shantanudhakadd/bank-customer-churn-prediction"
download_path = "/home/claude/data"

try:
    # Create data directory
    os.makedirs(download_path, exist_ok=True)

    # Download dataset
    print(f"Downloading dataset: {dataset_name}")
    os.system(f"kaggle datasets download -d {dataset_name} -p {download_path} --unzip")

    print("✓ Dataset downloaded successfully!")

    # List downloaded files
    files = os.listdir(download_path)
    print(f"Downloaded files: {files}")

except Exception as e:
    print(f"⚠ Error downloading from Kaggle: {e}")
    print("\nNote: To use Kaggle API, you need to:")
    print("1. Create a Kaggle account at kaggle.com")
    print("2. Go to Account settings -> API -> Create New API Token")
    print("3. Place kaggle.json in ~/.kaggle/ directory")
    print("\nAlternatively, download the dataset manually from:")
    print(f"https://www.kaggle.com/datasets/{dataset_name}")


[STEP 1] Downloading dataset from Kaggle...
✓ Kaggle API is installed
✓ Dataset downloaded successfully!
Downloaded files: ['Churn_Modelling.csv']


In [19]:
print("\n[STEP 2] Loading and exploring data...")

# Find the CSV file
csv_files = [f for f in os.listdir(download_path) if f.endswith('.csv')]
if not csv_files:
    print("⚠ No CSV file found. Please download the dataset manually.")
    exit()

data_file = os.path.join(download_path, csv_files[0])
print(f"Loading file: {csv_files[0]}")

df = pd.read_csv(data_file)

print(f"\n✓ Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

print("\n" + "="*70)
print("DATASET OVERVIEW")
print("="*70)
print("\nFirst 5 rows:")
print(df.head())

print("\n\nDataset Info:")
print(df.info())

print("\n\nStatistical Summary:")
print(df.describe())

print("\n\nMissing Values:")
missing = df.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("✓ No missing values found!")

print("\n\nTarget Variable Distribution:")
if 'Exited' in df.columns:
    churn_counts = df['Exited'].value_counts()
    print(churn_counts)
    print(f"\nChurn Rate: {(churn_counts[1] / len(df)) * 100:.2f}%")


[STEP 2] Loading and exploring data...
Loading file: Churn_Modelling.csv

✓ Dataset loaded successfully!
Shape: 10000 rows × 14 columns

DATASET OVERVIEW

First 5 rows:
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1         

In [20]:


print("\n[STEP 3] Performing Exploratory Data Analysis...")

# Create output directory for plots
os.makedirs('/home/claude/plots', exist_ok=True)

# Figure 1: Target Variable Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'Exited' in df.columns:
    # Count plot
    df['Exited'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71', '#e74c3c'])
    axes[0].set_title('Churn Distribution', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Exited (0=No, 1=Yes)')
    axes[0].set_ylabel('Count')
    axes[0].set_xticklabels(['Retained', 'Churned'], rotation=0)

    # Pie chart
    churn_counts = df['Exited'].value_counts()
    axes[1].pie(churn_counts, labels=['Retained', 'Churned'], autopct='%1.1f%%',
                colors=['#2ecc71', '#e74c3c'], startangle=90)
    axes[1].set_title('Churn Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('/home/claude/plots/1_target_distribution.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 1_target_distribution.png")
plt.close()

# Figure 2: Numerical Features Distribution
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Exited' in numerical_cols:
    numerical_cols.remove('Exited')

# Remove ID columns
numerical_cols = [col for col in numerical_cols if 'id' not in col.lower() and 'customer' not in col.lower()]

if len(numerical_cols) > 0:
    n_cols = 3
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes

    for idx, col in enumerate(numerical_cols):
        if idx < len(axes):
            df[col].hist(bins=30, ax=axes[idx], color='skyblue', edgecolor='black')
            axes[idx].set_title(f'{col} Distribution', fontweight='bold')
            axes[idx].set_xlabel(col)
            axes[idx].set_ylabel('Frequency')

    # Hide extra subplots
    for idx in range(len(numerical_cols), len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    plt.savefig('/home/claude/plots/2_numerical_distributions.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: 2_numerical_distributions.png")
    plt.close()

# Figure 3: Churn by Categorical Features
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

if len(categorical_cols) > 0 and 'Exited' in df.columns:
    n_cats = len(categorical_cols)
    fig, axes = plt.subplots(1, min(n_cats, 3), figsize=(15, 5))
    if n_cats == 1:
        axes = [axes]

    for idx, col in enumerate(categorical_cols[:3]):
        churn_by_cat = pd.crosstab(df[col], df['Exited'], normalize='index') * 100
        churn_by_cat.plot(kind='bar', stacked=True, ax=axes[idx],
                         color=['#2ecc71', '#e74c3c'])
        axes[idx].set_title(f'Churn Rate by {col}', fontweight='bold')
        axes[idx].set_xlabel(col)
        axes[idx].set_ylabel('Percentage')
        axes[idx].legend(['Retained', 'Churned'])
        axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45)

    plt.tight_layout()
    plt.savefig('/home/claude/plots/3_churn_by_categories.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: 3_churn_by_categories.png")
    plt.close()

# Figure 4: Correlation Heatmap
if len(numerical_cols) > 0:
    fig, ax = plt.subplots(figsize=(12, 10))

    corr_cols = numerical_cols + (['Exited'] if 'Exited' in df.columns else [])
    correlation = df[corr_cols].corr()

    sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, ax=ax, cbar_kws={'shrink': 0.8})
    ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)

    plt.tight_layout()
    plt.savefig('/home/claude/plots/4_correlation_heatmap.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: 4_correlation_heatmap.png")
    plt.close()


[STEP 3] Performing Exploratory Data Analysis...
✓ Saved: 1_target_distribution.png
✓ Saved: 2_numerical_distributions.png
✓ Saved: 3_churn_by_categories.png
✓ Saved: 4_correlation_heatmap.png


In [21]:
print("\n[STEP 4] Preprocessing data...")

# Create a copy for preprocessing
df_processed = df.copy()

# Remove unnecessary columns (IDs, names, etc.)
cols_to_drop = []
for col in df_processed.columns:
    if any(x in col.lower() for x in ['id', 'customer', 'surname', 'rownumber']):
        cols_to_drop.append(col)

if cols_to_drop:
    df_processed = df_processed.drop(cols_to_drop, axis=1)
    print(f"Dropped columns: {cols_to_drop}")

# Encode categorical variables
label_encoders = {}
categorical_cols = df_processed.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col])
    label_encoders[col] = le
    print(f"✓ Encoded: {col}")

# Separate features and target
if 'Exited' in df_processed.columns:
    X = df_processed.drop('Exited', axis=1)
    y = df_processed['Exited']
else:
    print("⚠ Warning: 'Exited' column not found. Please check your dataset.")
    exit()

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features scaled using StandardScaler")


[STEP 4] Preprocessing data...
Dropped columns: ['RowNumber', 'CustomerId', 'Surname']
✓ Encoded: Geography
✓ Encoded: Gender

Features shape: (10000, 10)
Target shape: (10000,)

Training set: 8000 samples
Test set: 2000 samples
✓ Features scaled using StandardScaler


In [23]:
print("\n[STEP 5] Training models...")
print("="*70)
# Store results
results = {}

# 1. LOGISTIC REGRESSION
print("\n[1] LOGISTIC REGRESSION")
print("-" * 50)

lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
roc_lr = roc_auc_score(y_test, y_pred_proba_lr)
f1_lr = f1_score(y_test, y_pred_lr)

results['Logistic Regression'] = {
    'model': lr_model,
    'predictions': y_pred_lr,
    'probabilities': y_pred_proba_lr,
    'accuracy': acc_lr,
    'roc_auc': roc_lr,
    'f1_score': f1_lr
}

print(f"Accuracy: {acc_lr:.4f}")
print(f"ROC-AUC: {roc_lr:.4f}")
print(f"F1-Score: {f1_lr:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['Retained', 'Churned']))

# 2. RANDOM FOREST
print("\n[2] RANDOM FOREST")
print("-" * 50)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
y_pred_proba_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
roc_rf = roc_auc_score(y_test, y_pred_proba_rf)
f1_rf = f1_score(y_test, y_pred_rf)

results['Random Forest'] = {
    'model': rf_model,
    'predictions': y_pred_rf,
    'probabilities': y_pred_proba_rf,
    'accuracy': acc_rf,
    'roc_auc': roc_rf,
    'f1_score': f1_rf
}

print(f"Accuracy: {acc_rf:.4f}")
print(f"ROC-AUC: {roc_rf:.4f}")
print(f"F1-Score: {f1_rf:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['Retained', 'Churned']))

# 3. GRADIENT BOOSTING
print("\n[3] GRADIENT BOOSTING")
print("-" * 50)

gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train_scaled, y_train)

y_pred_gb = gb_model.predict(X_test_scaled)
y_pred_proba_gb = gb_model.predict_proba(X_test_scaled)[:, 1]

acc_gb = accuracy_score(y_test, y_pred_gb)
roc_gb = roc_auc_score(y_test, y_pred_proba_gb)
f1_gb = f1_score(y_test, y_pred_gb)

results['Gradient Boosting'] = {
    'model': gb_model,
    'predictions': y_pred_gb,
    'probabilities': y_pred_proba_gb,
    'accuracy': acc_gb,
    'roc_auc': roc_gb,
    'f1_score': f1_gb
}

print(f"Accuracy: {acc_gb:.4f}")
print(f"ROC-AUC: {roc_gb:.4f}")
print(f"F1-Score: {f1_gb:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_gb, target_names=['Retained', 'Churned']))


[STEP 5] Training models...

[1] LOGISTIC REGRESSION
--------------------------------------------------
Accuracy: 0.8050
ROC-AUC: 0.7710
F1-Score: 0.2292

Classification Report:
              precision    recall  f1-score   support

    Retained       0.82      0.97      0.89      1593
     Churned       0.59      0.14      0.23       407

    accuracy                           0.81      2000
   macro avg       0.70      0.56      0.56      2000
weighted avg       0.77      0.81      0.75      2000


[2] RANDOM FOREST
--------------------------------------------------
Accuracy: 0.8645
ROC-AUC: 0.8469
F1-Score: 0.5798

Classification Report:
              precision    recall  f1-score   support

    Retained       0.88      0.97      0.92      1593
     Churned       0.79      0.46      0.58       407

    accuracy                           0.86      2000
   macro avg       0.83      0.71      0.75      2000
weighted avg       0.86      0.86      0.85      2000


[3] GRADIENT BOOSTING


In [24]:


# Create comparison table
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results.keys()],
    'ROC-AUC': [results[m]['roc_auc'] for m in results.keys()],
    'F1-Score': [results[m]['f1_score'] for m in results.keys()]
})

print("\nMODEL COMPARISON")
print(comparison_df.to_string(index=False))

# Find best model
best_model_name = comparison_df.loc[comparison_df['ROC-AUC'].idxmax(), 'Model']
print(f"\n🏆 Best Model (by ROC-AUC): {best_model_name}")

# Save comparison table
comparison_df.to_csv('/home/claude/model_comparison.csv', index=False)
print("\n✓ Saved: model_comparison.csv")


MODEL COMPARISON
              Model  Accuracy  ROC-AUC  F1-Score
Logistic Regression    0.8050 0.771044  0.229249
      Random Forest    0.8645 0.846900  0.579845
  Gradient Boosting    0.8675 0.867301  0.594181

🏆 Best Model (by ROC-AUC): Gradient Boosting

✓ Saved: model_comparison.csv


In [25]:

print("\n[STEP 7] Creating performance visualizations...")

# Figure 5: Model Comparison Bar Chart
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metrics = ['Accuracy', 'ROC-AUC', 'F1-Score']
for idx, metric in enumerate(metrics):
    comparison_df.plot(x='Model', y=metric, kind='bar', ax=axes[idx],
                       color='skyblue', legend=False)
    axes[idx].set_title(f'{metric} Comparison', fontsize=14, fontweight='bold')
    axes[idx].set_ylabel(metric)
    axes[idx].set_xlabel('')
    axes[idx].set_ylim([0, 1])
    axes[idx].tick_params(axis='x', rotation=45)

    # Add value labels on bars
    for container in axes[idx].containers:
        axes[idx].bar_label(container, fmt='%.3f')

plt.tight_layout()
plt.savefig('/home/claude/plots/5_model_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 5_model_comparison.png")
plt.close()

# Figure 6: ROC Curves
fig, ax = plt.subplots(figsize=(10, 8))

colors = ['#3498db', '#e74c3c', '#2ecc71']
for idx, (model_name, model_data) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, model_data['probabilities'])
    ax.plot(fpr, tpr, label=f"{model_name} (AUC = {model_data['roc_auc']:.3f})",
            linewidth=2, color=colors[idx])

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves Comparison', fontsize=16, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/claude/plots/6_roc_curves.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 6_roc_curves.png")
plt.close()

# Figure 7: Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (model_name, model_data) in enumerate(results.items()):
    cm = confusion_matrix(y_test, model_data['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['Retained', 'Churned'],
                yticklabels=['Retained', 'Churned'])
    axes[idx].set_title(f'{model_name}\nConfusion Matrix', fontweight='bold')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('/home/claude/plots/7_confusion_matrices.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 7_confusion_matrices.png")
plt.close()

# Figure 8: Feature Importance (for tree-based models)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Random Forest Feature Importance
feature_importance_rf = pd.DataFrame({
    'Feature': X.columns,
    'Importance': results['Random Forest']['model'].feature_importances_
}).sort_values('Importance', ascending=False).head(10)

axes[0].barh(feature_importance_rf['Feature'], feature_importance_rf['Importance'], color='coral')
axes[0].set_xlabel('Importance')
axes[0].set_title('Top 10 Features - Random Forest', fontweight='bold')
axes[0].invert_yaxis()

# Gradient Boosting Feature Importance
feature_importance_gb = pd.DataFrame({
    'Feature': X.columns,
    'Importance': results['Gradient Boosting']['model'].feature_importances_
}).sort_values('Importance', ascending=False).head(10)

axes[1].barh(feature_importance_gb['Feature'], feature_importance_gb['Importance'], color='lightgreen')
axes[1].set_xlabel('Importance')
axes[1].set_title('Top 10 Features - Gradient Boosting', fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('/home/claude/plots/8_feature_importance.png', dpi=300, bbox_inches='tight')
print("✓ Saved: 8_feature_importance.png")
plt.close()


[STEP 7] Creating performance visualizations...
✓ Saved: 5_model_comparison.png
✓ Saved: 6_roc_curves.png
✓ Saved: 7_confusion_matrices.png
✓ Saved: 8_feature_importance.png


In [26]:

print("\n[STEP 8] Saving results and summary...")

# Create comprehensive summary report
summary = f"""
{'='*70}
CUSTOMER CHURN PREDICTION - SUMMARY REPORT
{'='*70}

DATASET INFORMATION:
-------------------
- Total Records: {len(df)}
- Features: {X.shape[1]}
- Training Samples: {len(X_train)}
- Test Samples: {len(X_test)}
- Churn Rate: {(y.sum() / len(y)) * 100:.2f}%

MODEL PERFORMANCE:
-----------------
"""

for model_name, model_data in results.items():
    summary += f"\n{model_name}:\n"
    summary += f"  - Accuracy:  {model_data['accuracy']:.4f}\n"
    summary += f"  - ROC-AUC:   {model_data['roc_auc']:.4f}\n"
    summary += f"  - F1-Score:  {model_data['f1_score']:.4f}\n"

summary += f"\nBEST MODEL: {best_model_name}\n"
summary += f"\nTOP 5 IMPORTANT FEATURES:\n"
summary += "-" * 40 + "\n"

if 'Random Forest' in results:
    top_features = feature_importance_rf.head(5)
    for idx, row in top_features.iterrows():
        summary += f"{row['Feature']:20s}: {row['Importance']:.4f}\n"

summary += f"\n{'='*70}\n"

# Save summary
with open('/home/claude/summary_report.txt', 'w') as f:
    f.write(summary)

print(summary)
print("✓ Saved: summary_report.txt")

# Save predictions
predictions_df = pd.DataFrame({
    'Actual': y_test.values,
    'LR_Prediction': results['Logistic Regression']['predictions'],
    'RF_Prediction': results['Random Forest']['predictions'],
    'GB_Prediction': results['Gradient Boosting']['predictions'],
    'LR_Probability': results['Logistic Regression']['probabilities'],
    'RF_Probability': results['Random Forest']['probabilities'],
    'GB_Probability': results['Gradient Boosting']['probabilities']
})

predictions_df.to_csv('/home/claude/predictions.csv', index=False)
print("✓ Saved: predictions.csv")

print("ANALYSIS COMPLETE!")
print("="*70)
print("\nGenerated Files:")
print("  📊 Visualizations (8 plots) -> /home/claude/plots/")
print("  📈 Model Comparison -> model_comparison.csv")
print("  🔮 Predictions -> predictions.csv")
print("  📄 Summary Report -> summary_report.txt")



[STEP 8] Saving results and summary...

CUSTOMER CHURN PREDICTION - SUMMARY REPORT

DATASET INFORMATION:
-------------------
- Total Records: 10000
- Features: 10
- Training Samples: 8000
- Test Samples: 2000
- Churn Rate: 20.37%

MODEL PERFORMANCE:
-----------------

Logistic Regression:
  - Accuracy:  0.8050
  - ROC-AUC:   0.7710
  - F1-Score:  0.2292

Random Forest:
  - Accuracy:  0.8645
  - ROC-AUC:   0.8469
  - F1-Score:  0.5798

Gradient Boosting:
  - Accuracy:  0.8675
  - ROC-AUC:   0.8673
  - F1-Score:  0.5942

BEST MODEL: Gradient Boosting

TOP 5 IMPORTANT FEATURES:
----------------------------------------
Age                 : 0.2399
EstimatedSalary     : 0.1471
CreditScore         : 0.1441
Balance             : 0.1412
NumOfProducts       : 0.1291


✓ Saved: summary_report.txt
✓ Saved: predictions.csv
ANALYSIS COMPLETE!

Generated Files:
  📊 Visualizations (8 plots) -> /home/claude/plots/
  📈 Model Comparison -> model_comparison.csv
  🔮 Predictions -> predictions.csv
  📄 Sum

In [27]:
print("\n[STEP 9] Test Your Own Customer")
print("Type 'exit' to stop\n")

feature_names = X.columns.tolist()
print("Enter values in this order:")
print(feature_names)

while True:
    user_input = input("\nEnter customer details separated by comma or 'exit': ")
    if user_input.lower() == 'exit':
        print("Exiting interactive testing.")
        break

    try:
        values = list(map(float, user_input.split(",")))
        if len(values) != len(feature_names):
            print(f"⚠ Please enter exactly {len(feature_names)} values.")
            continue

        # Scale features
        input_scaled = scaler.transform([values])

        for model_name, model_data in results.items():
            pred = model_data['model'].predict(input_scaled)[0]
            prob = model_data['model'].predict_proba(input_scaled)[0][1]
            print(f"\n{model_name}:")
            print("Prediction:", "CHURN 🚨" if pred==1 else "STAYED ✅")
            print(f"Churn Probability: {prob*100:.2f}%")
        print("-"*50)

    except ValueError:
        print("⚠ Invalid input. Make sure all values are numeric and separated by commas.")



[STEP 9] Test Your Own Customer
Type 'exit' to stop

Enter values in this order:
['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

Enter customer details separated by comma or 'exit': 650, 0, 0, 35, 5, 50000, 2, 1, 1, 60000

Logistic Regression:
Prediction: STAYED ✅
Churn Probability: 8.43%

Random Forest:
Prediction: STAYED ✅
Churn Probability: 1.00%

Gradient Boosting:
Prediction: STAYED ✅
Churn Probability: 4.94%
--------------------------------------------------

Enter customer details separated by comma or 'exit': 820, 0, 1, 48, 10, 150000, 4, 1, 1, 120000

Logistic Regression:
Prediction: STAYED ✅
Churn Probability: 14.93%

Random Forest:
Prediction: CHURN 🚨
Churn Probability: 94.00%

Gradient Boosting:
Prediction: CHURN 🚨
Churn Probability: 95.29%
--------------------------------------------------

Enter customer details separated by comma or 'exit': exit
Exiting interactive testing.


In [15]:
test_samples = [
    [650, 0, 0, 35, 5, 50000, 2, 1, 1, 60000],
    [450, 1, 1, 55, 1, 0, 1, 0, 0, 35000],
    [780, 0, 1, 45, 9, 130000, 3, 1, 1, 100000],
    [600, 2, 0, 40, 3, 70000, 1, 1, 0, 55000],
    [520, 1, 0, 50, 2, 20000, 1, 1, 0, 40000],
    [710, 0, 1, 30, 7, 90000, 2, 1, 1, 85000],
    [480, 2, 1, 60, 1, 0, 1, 0, 0, 30000],
    [690, 0, 0, 38, 6, 60000, 2, 1, 1, 65000],
    [580, 1, 1, 42, 4, 80000, 1, 1, 0, 50000],
    [820, 0, 1, 48, 10, 150000, 4, 1, 1, 120000]
]
